# miniKanren (relational programming)

**Domain:** Symbolic AI & Logic  ·  **recommended addition**  ·  **runnable:** yes

A compact refresher on miniKanren — the tiny relational-programming language you embed
in a host language to *search for values that satisfy a set of relations*, running the
same program forwards, backwards, or both at once.

## 1. What & Why

**What it is.** miniKanren is a small, embeddable logic-programming language. You write
**relations** (not functions) and ask the system to find every assignment of *logic
variables* that makes those relations simultaneously true. The whole language is a handful
of operators — historically implemented in ~50 lines of Scheme — and has been ported to
dozens of host languages. In Python the canonical port is **`kanren`** (a.k.a. `logpy`).

**The problem it solves.** A normal function runs in one direction: inputs → output.
A *relation* has no privileged direction. `append(xs, ys) == zs` is just a statement about
three lists; given any of the three you can solve for the others. miniKanren turns that
"statement about values" into a runnable search: declare the relation once, then query it
with knowns or unknowns in *any* position.

**When to reach for it.**
- You want **multiple solutions** to a constraint, not one answer (enumerate all paths,
  all type derivations, all puzzle solutions).
- You want **bidirectional** computation — synthesis from a spec, or "run the interpreter
  backwards" to find programs that produce a result (the *quine*/program-synthesis demos).
- You're prototyping a **rules engine** or **pattern matcher** and want unification +
  backtracking without standing up a full Prolog.

**When not to.** It is a *search*, not a *solver*: there's no constraint propagation or
clever heuristics by default, so large combinatorial problems blow up. For heavy
numeric/finite-domain constraints use a real CP/SMT solver ([`z3-smt`](z3-smt.ipynb),
[`minizinc`](minizinc.ipynb)); for a mature deductive database use
[`datalog`](datalog.ipynb) or [`swi-prolog`](swi-prolog.ipynb).

## 2. Mental Model

> **Spreadsheet of unknowns + a substitution that grows by unification.**

Picture every logic variable as an empty cell. A goal like `eq(x, 5)` says "cell `x`
must equal 5". The engine maintains a **substitution** — the partial map of cells that
have been pinned down so far — and tries to extend it by **unifying** the two sides of
each goal:

- Unifying a fresh variable with a term **binds** it (writes the cell).
- Unifying two ground terms **succeeds** if they're already equal, else the branch **fails**.
- Goals are combined with **conjunction** (`lall` / a `conde` row: *all* must hold) and
  **disjunction** (`conde` across rows: *try each* alternative, branching the search).

`run(n, x, *goals)` walks that search tree lazily and hands back the first `n` values of
`x` consistent with a complete substitution (`run(0, ...)` means "all of them"). Because
the cells aren't typed as input or output, putting a variable where you'd normally put an
argument is exactly what makes the program run "backwards".

## 3. Key Concepts

| Term | What it means |
| --- | --- |
| **Logic variable** | A hole to be solved for. Create with `x = var()` (or `var('name')`). Unbound until the search binds it. |
| **Unification** | The core operation: make two terms equal by binding variables. `eq(a, b)` is the unify goal. Symmetric — neither side is "the input". |
| **Substitution** | The current partial answer: which variables are bound to what. The search threads and extends it. |
| **Goal** | A relation applied to terms, e.g. `membero(x, (1,2,3))`. Evaluates against a substitution to a (lazy) stream of extended substitutions — empty stream = failure. |
| **`run(n, var, *goals)`** | Run the conjunction of `goals`, return up to `n` reified values of `var`. `n=0` → every solution. |
| **`eq`** | Unify two terms. The workhorse. |
| **`conde`** | Disjunction of conjunctions: `conde([g1, g2], [g3])` = "(g1 ∧ g2) ∨ g3". Each row is an alternative branch. |
| **`lall` / `lany`** | Explicit conjunction (`all`) and disjunction (`any`) of goals. |
| **`membero(x, coll)`** | Relational membership — non-deterministically binds `x` to each element. |
| **`Relation` / `facts`** | A user-defined relation plus a table of ground tuples (a mini knowledge base) you can query in any direction. |
| **Reification** | Turning the internal substitution back into a concrete answer term for `var` — what `run` returns. |

**The golden rule:** you define *relations*, you query them with `run`, and any argument
position may be a known value **or** a logic variable.

## 4. Setup

Pure Python, no native deps, tiny install. The package is `kanren` (also published as
`logpy`); it pulls in `unification` and `multipledispatch`.

In [ ]:
# %pip install kanren   # uncomment on a fresh kernel
import kanren
print("kanren", kanren.__version__)

## 5. Worked Examples

Three escalating examples: (1) the fundamentals — variables, unification, multiple
answers; (2) a small **knowledge base** queried *in both directions*; (3) a logic puzzle
solved by generate-and-constrain.

### Example 1 — Variables, unification, and multiple answers

`run(n, q, *goals)` solves the conjunction of `goals` and returns up to `n` values of the
query variable `q`. Note how `membero` yields *every* consistent binding, and how a single
`eq` between two structured terms solves for the holes inside.

In [ ]:
from kanren import run, eq, var, membero, conde, lall

x = var()

# eq is unification: pin x to a value.
print("eq:        ", run(1, x, eq(x, 5)))

# membero enumerates every element that could be x  -> three solutions.
print("membero:   ", run(0, x, membero(x, (1, 2, 3))))

# run(0, ...) means "all solutions"; conde is disjunction.
print("conde:     ", run(0, x, conde([eq(x, "a")], [eq(x, "b")])))

# Unification looks *inside* structures and solves for the holes.
a, b = var(), var()
print("structural:", run(1, (a, b), eq((1, a), (b, 2))))   # forces a=2, b=1


### Example 2 — A knowledge base, queried both directions

This is the heart of relational programming. We declare one `parent` relation as a table
of facts, then ask questions by leaving **different positions unknown** — children of a
node, parents of a node, and (by chaining with a shared intermediate variable) grandparents
— all from the *same* relation, no separate "reverse" function needed.

In [ ]:
from kanren import Relation, facts, run, var, conde

parent = Relation()
facts(parent,
      ("Abe", "Homer"),
      ("Homer", "Bart"),
      ("Homer", "Lisa"),
      ("Marge", "Bart"),
      ("Marge", "Lisa"))

child, who = var(), var()

# Forward: who are Homer's children?  (unknown in the 2nd position)
print("Homer's kids:   ", run(0, child, parent("Homer", child)))

# Backward: who are Bart's parents?   (unknown in the 1st position)
print("Bart's parents: ", run(0, who, parent(who, "Bart")))

# Compose relations to define a new one: grandparent.
def grandparent(gp, gc):
    p = var()
    return conde([parent(gp, p), parent(p, gc)])

gp = var()
print("Bart's grandparents:", run(0, gp, grandparent(gp, "Bart")))


### Example 3 — A logic puzzle (generate-and-constrain)

Three people live in houses numbered 1–3, one each. Alice is *not* in house 1, and Bob is
in house 2. We let `permuteq` generate the distinct assignments and let the remaining goals
prune them. `run` returns only the assignments that satisfy every constraint at once — this
is exactly how you'd express small scheduling/seating/coloring problems.

In [ ]:
from kanren import run, var, lall, eq, membero, permuteq

alice, bob, carol = var(), var(), var()

solutions = run(0, (alice, bob, carol), lall(
    permuteq((alice, bob, carol), (1, 2, 3)),  # distinct houses 1..3
    membero(alice, (2, 3)),                    # Alice is not in house 1
    eq(bob, 2),                                # Bob is in house 2
))

for alice_h, bob_h, carol_h in solutions:
    print(f"Alice={alice_h}  Bob={bob_h}  Carol={carol_h}")
print("total solutions:", len(solutions))


## 6. Gotchas & Pitfalls

- **`run(0, ...)` means *all* solutions — and it's eager.** With recursive relations or
  unbounded generators that stream is infinite; `run(0, ...)` will hang. Ask for a finite
  count (`run(5, ...)`) until you know the search terminates.
- **Goal order changes termination, not just speed.** miniKanren's interleaving search is
  forgiving, but a recursive goal placed *before* the goal that grounds its arguments can
  diverge. Put the "generator" (e.g. `membero`, `permuteq`) before the "filter" goals.
- **It searches; it does not propagate constraints.** There's no finite-domain
  arc-consistency. A puzzle a CP solver cracks instantly can blow up combinatorially here.
  miniKanren shines for *expressiveness and multiple answers*, not raw constraint horsepower.
- **Relations, not functions.** Don't `return` a Python value from inside a relation and
  expect the engine to see it — relations return **goals**. Build answers by *unifying* a
  query variable (`eq(q, ...)`), not by computing and returning.
- **No built-in disequality / negation in the base package.** There's no plain `neq` in
  `kanren` 0.2.x. "All different" is easiest via `permuteq` over a fixed domain; richer
  constraint logic (`=/=`, `symbolo`, `numbero`) lives in fuller miniKanren dialects
  (e.g. Racket/Scheme `faster-miniKanren`, or `core.logic` in Clojure).
- **Fresh variables are per-call.** Create new `var()`s inside recursive relation
  definitions so each level of the search gets its own holes; reusing one variable across
  branches silently over-constrains the search.

## 7. When to Use vs Alternatives

| Option | Best when | vs miniKanren |
| --- | --- | --- |
| **miniKanren (`kanren`)** | You want relational/bidirectional search *embedded in Python*, multiple answers, program synthesis, light rule systems. | Tiny, pure-Python, no DSL boundary — but a naive search with no constraint propagation. |
| **Prolog** ([`swi-prolog`](swi-prolog.ipynb)) | You want a mature, fast logic engine with cut, indexing, libraries, and a real REPL. | Far more performant and battle-tested; separate language/runtime instead of a Python embedding. |
| **Datalog** ([`datalog`](datalog.ipynb)) | Recursive *queries over data* that must terminate and scale (graph reachability, lineage). | Decidable and bottom-up; no general unification of structured terms, but predictable and fast. |
| **SMT / CP solvers** ([`z3-smt`](z3-smt.ipynb), [`minizinc`](minizinc.ipynb)) | Hard combinatorial or arithmetic constraints needing real propagation & optimization. | Vastly stronger search; you model declaratively but lose the "just run the relation backwards" programming style. |
| **ASP** ([`answer-set-programming`](answer-set-programming.ipynb)) | Combinatorial search with negation-as-failure and stable-model semantics. | Grounded + solved efficiently; different paradigm from on-the-fly relational search. |

**Rule of thumb:** reach for miniKanren to *think relationally inside Python* and to get
many answers cheaply on small problems. Graduate to Prolog/Datalog/SMT/ASP the moment
performance, scale, or rich constraints dominate.

## 8. Resources

- **The miniKanren home page** — papers, talks, and the canonical list of implementations:
  <http://minikanren.org/>
- ***The Reasoned Schemer* (2nd ed.)**, Friedman, Byrd, Kiselyov, Hemann — the book that
  teaches miniKanren from `eq`/`conde` upward: <https://mitpress.mit.edu/9780262535519/the-reasoned-schemer/>
- **`kanren` (Python port) on GitHub** — source, examples, and the unification engine it
  builds on: <https://github.com/pythological/kanren>
- **William Byrd, "miniKanren in 100 lines" / the relational interpreter & quine demos**
  (strange-loop talk): <https://www.youtube.com/watch?v=RVDCRlW1f1Y>
- **`core.logic`** — the influential Clojure port, useful for seeing the same ideas with
  disequality and finite-domain constraints: <https://github.com/clojure/core.logic>

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def solve_append(xs, ys, zs):
    """Every (xs, ys, zs) with xs + ys == zs consistent with what is known."""
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE